# Token Agent Mixed Benchmark 数据格式查看

查看 `preprocess_data.sh` 处理后生成的 `train.parquet` / `test.parquet` 的数据格式和内容。

In [3]:
import json
import pandas as pd

DATA_DIR = "/mnt/workspace/wxc/roleagent/data/token_agent_mixed"

train_df = pd.read_parquet(f"{DATA_DIR}/train.parquet")
test_df  = pd.read_parquet(f"{DATA_DIR}/test.parquet")

print(f"Train rows: {len(train_df):,}")
print(f"Test  rows: {len(test_df):,}")
print(f"\nColumns: {train_df.columns.tolist()}")

Train rows: 272,431
Test  rows: 73,016

Columns: ['data_source', 'task_category', 'prompt', 'env_kwargs', 'reward_model', 'extra_info']


## 1. 各数据集行数分布

In [4]:
print("=== Train split ===")
print(train_df["data_source"].value_counts().to_string())
print("\n=== Test split ===")
print(test_df["data_source"].value_counts().to_string())

=== Train split ===
data_source
hotpotqa          90447
squad             87599
nq                79168
lighteval/MATH     7500
openai/gsm8k       7473
math_dapo           244

=== Test split ===
data_source
popqa              14267
2wikimultihopqa    12576
triviaqa           11313
squad              10570
hotpotqa            7405
lighteval/MATH      5000
simpleqa            4326
nq                  3610
musique             2417
openai/gsm8k        1319
bamboogle            125
aime_2024             30
aime_2025             30
math_dapo             28


## 2. task_category 分布

In [5]:
CATEGORY_NAMES = {
    0: "math_reasoning",
    1: "quick_qa",
    2: "direct_qa",
    3: "search_qa",
    4: "action_env",
}

print("=== Train task_category ===")
for cat_id, count in train_df["task_category"].value_counts().sort_index().items():
    print(f"  {cat_id} ({CATEGORY_NAMES.get(cat_id, '?')}): {count:,}")

print("\n=== Test task_category ===")
for cat_id, count in test_df["task_category"].value_counts().sort_index().items():
    print(f"  {cat_id} ({CATEGORY_NAMES.get(cat_id, '?')}): {count:,}")

=== Train task_category ===
  0 (math_reasoning): 15,217
  1 (quick_qa): 87,599
  3 (search_qa): 169,615

=== Test task_category ===
  0 (math_reasoning): 6,407
  1 (quick_qa): 10,570
  2 (direct_qa): 4,326
  3 (search_qa): 51,713


## 3. 查看单条记录的完整格式

In [6]:
def show_row(df, data_source, idx=0):
    """打印指定 data_source 的第 idx 条记录，反序列化 JSON 字段。"""
    subset = df[df["data_source"] == data_source]
    if len(subset) == 0:
        print(f"No rows for data_source={data_source!r}")
        return
    row = subset.iloc[idx].to_dict()
    # 反序列化 JSON 字符串字段
    for col in ["env_kwargs", "reward_model"]:
        if isinstance(row.get(col), str):
            try:
                row[col] = json.loads(row[col])
            except Exception:
                pass
    print(f"\n{'='*60}")
    print(f"data_source  : {row['data_source']}")
    print(f"task_category: {row['task_category']} ({CATEGORY_NAMES.get(row['task_category'], '?')})")
    print(f"\n--- prompt ({len(row['prompt'])} messages) ---")
    for msg in row["prompt"]:
        role = msg.get("role", "?")
        content = msg.get("content", "")
        preview = content[:300] + "..." if len(content) > 300 else content
        print(f"  [{role}] {preview}")
    print(f"\n--- env_kwargs ---")
    print(json.dumps(row["env_kwargs"], ensure_ascii=False, indent=2)[:800])
    print(f"\n--- reward_model ---")
    print(json.dumps(row["reward_model"], ensure_ascii=False, indent=2)[:400] if row["reward_model"] else "None")
    print(f"\n--- extra_info ---")
    print(row["extra_info"])

### 3.1 GSM8K（数学推理）

In [7]:
show_row(train_df, "openai/gsm8k")


data_source  : openai/gsm8k
task_category: 0 (math_reasoning)

--- prompt (2 messages) ---
  [system] You are a versatile AI assistant capable of solving a wide range of tasks. You have access to the following tools and capabilities:

## Tools

### Search
Use the search tool to look up external information when needed.
Format: <search>your query</search>
The search result will be returned inside <in...
  [user] Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

--- env_kwargs ---
{
  "ground_truth": "Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72",
  "question": "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?",
  "data_source": "openai/gsm8k",
  "task_category": 0
}

--- reward_model ---
{
  "

### 3.2 MATH（竞赛数学）

In [8]:
show_row(train_df, "lighteval/MATH")


data_source  : lighteval/MATH
task_category: 0 (math_reasoning)

--- prompt (2 messages) ---
  [system] You are a versatile AI assistant capable of solving a wide range of tasks. You have access to the following tools and capabilities:

## Tools

### Search
Use the search tool to look up external information when needed.
Format: <search>your query</search>
The search result will be returned inside <in...
  [user] Let \[f(x) = \left\{
\begin{array}{cl} ax+3, &\text{ if }x>2, \\
x-5 &\text{ if } -2 \le x \le 2, \\
2x-b &\text{ if } x <-2.
\end{array}
\right.\]Find $a+b$ if the piecewise function is continuous (which means that its graph can be drawn without lifting your pencil from the paper).

--- env_kwargs ---
{
  "ground_truth": "For the piecewise function to be continuous, the cases must \"meet\" at $2$ and $-2$. For example, $ax+3$ and $x-5$ must be equal when $x=2$. This implies $a(2)+3=2-5$, which we solve to get $2a=-6 \\Rightarrow a=-3$. Similarly, $x-5$ and $2x-b$ must be equ

### 3.3 SQuAD（阅读理解）

In [9]:
show_row(train_df, "squad")


data_source  : squad
task_category: 1 (quick_qa)

--- prompt (2 messages) ---
  [system] You are a versatile AI assistant capable of solving a wide range of tasks. You have access to the following tools and capabilities:

## Tools

### Search
Use the search tool to look up external information when needed.
Format: <search>your query</search>
The search result will be returned inside <in...
  [user] Context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Bui...

--- env_kwargs ---
{
  "ground_truth": [
    "Saint Bernadette Soubirous"
  ],
  "question": "Context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a

### 3.4 Search QA（搜索问答，查看子数据集分布）

In [10]:
# search_qa 子数据集分布
search_sources = ["nq", "trivia_qa", "hotpotqa", "2wikimultihopqa", "musique", "bamboogle", "popqa"]
print("=== Search QA 子数据集（train）===")
for src in search_sources:
    count = (train_df["data_source"] == src).sum()
    if count > 0:
        print(f"  {src}: {count:,}")

# 显示实际存在的 search 相关 data_source
all_sources = train_df["data_source"].unique().tolist()
print("\n所有 data_source:", all_sources)

=== Search QA 子数据集（train）===
  nq: 79,168
  hotpotqa: 90,447

所有 data_source: ['openai/gsm8k', 'lighteval/MATH', 'math_dapo', 'squad', 'nq', 'hotpotqa']


In [11]:
# 取第一个 search 相关的子数据集展示
search_df = train_df[~train_df["data_source"].isin(["openai/gsm8k", "lighteval/MATH", "math_dapo", "squad", "simpleqa", "alfworld", "webshop", "sokoban", "ezpoints"])]
if len(search_df) > 0:
    first_src = search_df["data_source"].iloc[0]
    show_row(train_df, first_src)


data_source  : nq
task_category: 3 (search_qa)

--- prompt (2 messages) ---
  [system] You are a versatile AI assistant capable of solving a wide range of tasks. You have access to the following tools and capabilities:

## Tools

### Search
Use the search tool to look up external information when needed.
Format: <search>your query</search>
The search result will be returned inside <in...
  [user] [{'content': 'You are a helpful and harmless assistant.', 'role': 'system'}
 {'content': 'total number of death row inmates in the us?', 'role': 'user'}]

--- env_kwargs ---
{
  "data_source": "nq",
  "ground_truth": {
    "target": "['2,718']"
  },
  "question": "total number of death row inmates in the us?"
}

--- reward_model ---
{
  "ground_truth": {
    "target": "['2,718']"
  },
  "style": "rule"
}

--- extra_info ---
{'index': 0, 'need_tools_kwargs': True, 'question': 'total number of death row inmates in the us?', 'split': 'train', 'tools_kwargs': {'search': {'create_kwargs': {'data_s

## 4. env_kwargs 字段结构对比（不同数据集）

In [12]:
print("各数据集 env_kwargs 的 key 结构：")
for src in train_df["data_source"].unique():
    sample = train_df[train_df["data_source"] == src].iloc[0]
    env_kw = sample["env_kwargs"]
    if isinstance(env_kw, str):
        try:
            env_kw = json.loads(env_kw)
        except Exception:
            pass
    keys = list(env_kw.keys()) if isinstance(env_kw, dict) else type(env_kw).__name__
    gt = env_kw.get("ground_truth", "N/A") if isinstance(env_kw, dict) else "N/A"
    gt_type = type(gt).__name__
    print(f"  {src:30s} keys={keys}  ground_truth type={gt_type}")

各数据集 env_kwargs 的 key 结构：
  openai/gsm8k                   keys=['ground_truth', 'question', 'data_source', 'task_category']  ground_truth type=str
  lighteval/MATH                 keys=['ground_truth', 'question', 'data_source', 'task_category']  ground_truth type=str
  math_dapo                      keys=['ground_truth', 'question', 'data_source', 'task_category']  ground_truth type=str
  squad                          keys=['ground_truth', 'question', 'data_source', 'task_category']  ground_truth type=list
  nq                             keys=['data_source', 'ground_truth', 'question']  ground_truth type=dict
  hotpotqa                       keys=['data_source', 'ground_truth', 'question']  ground_truth type=dict


## 5. prompt 格式示例（原始 chat messages）

In [13]:
row = train_df.iloc[0]
print(f"data_source: {row['data_source']}")
print(f"prompt 类型: {type(row['prompt']).__name__}, 消息数: {len(row['prompt'])}")
for msg in row["prompt"]:
    print(f"\n  role: {msg['role']}")
    print(f"  content (前500字): {msg['content'][:500]}")

data_source: openai/gsm8k
prompt 类型: ndarray, 消息数: 2

  role: system
  content (前500字): You are a versatile AI assistant capable of solving a wide range of tasks. You have access to the following tools and capabilities:

## Tools

### Search
Use the search tool to look up external information when needed.
Format: <search>your query</search>
The search result will be returned inside <information>...</information> tags.

### Environment Actions
When operating in an interactive environment, you may take actions.
Format: <action>your action</action>

Available environment actions inclu

  role: user
  content (前500字): Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
